In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# -----------------------------
# Config
# -----------------------------
ALPHA = 0.10
R = 100
TRAIN_FRAC = 0.70
SEED = 42

ELEC2_PATH = Path("../data/electricity-normalized.csv")

COVARIATE_COLS = ["nswprice", "nswdemand", "vicprice", "vicdemand"]
RESPONSE_COL = "transfer"

# remove initial constant-transfer stretch.
REMOVE_INITIAL = 17760

# For full half-hour ELEC2, 48 observations = about 1 day.
BOOT_BLOCK_LEN = 48
BOOT_B = 1000

In [ ]:
# =========================================================
# 1) Load and process ELEC2
# =========================================================

def load_elec2_arrays(
    path=ELEC2_PATH,
    covariate_cols=COVARIATE_COLS,
    response_col=RESPONSE_COL,
    remove_initial=REMOVE_INITIAL,
    keep_full_series=True,
):
    """
    Load ELEC2 normalized CSV and return:
      X: covariates
      y: response
      df: processed DataFrame
      
      - remove first 17760 rows
      - use full remaining time series
      - covariates: nswprice, nswdemand, vicprice, vicdemand
      - response: transfer
    """
    df = pd.read_csv(path)

    missing = [c for c in covariate_cols + [response_col] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in ELEC2 file: {missing}\nAvailable columns: {list(df.columns)}")

    if remove_initial is not None and remove_initial > 0:
        df = df.iloc[remove_initial:].reset_index(drop=True)

    # Keep full series by default.
    # If later you want 9am-12pm only, filter df here.
    if not keep_full_series:
        raise NotImplementedError("This version keeps the full ELEC2 series, matching your current notebook.")

    X = df[covariate_cols].to_numpy(dtype=np.float64)
    y = df[response_col].to_numpy(dtype=np.float64)

    # Useful time index for plotting / alignment
    df = df.copy()
    df["time_idx"] = np.arange(len(df))

    return X, y, df


X_elec, y_elec, elec_df = load_elec2_arrays()

N, p = X_elec.shape
train_end = int(TRAIN_FRAC * N)

print("ELEC2 loaded:")
print("  X shape:", X_elec.shape)
print("  y shape:", y_elec.shape)
print("  train_end:", train_end)
print("  test rows:", N - train_end)
print("  covariates:", COVARIATE_COLS)

In [ ]:
# =========================================================
# 2) Fit fixed GBRT model
# =========================================================

from sklearn.ensemble import HistGradientBoostingRegressor

def fit_gbrt_model(X_train, y_train, seed=42):
    """
    Fixed GBRT point predictor.
    """
    model = HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        random_state=seed,
        l2_regularization=0.0,
    )
    model.fit(X_train, y_train)
    return model


def mse(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.mean((a - b) ** 2))


t0 = time.perf_counter()

gbrt = fit_gbrt_model(
    X_elec[:train_end],
    y_elec[:train_end],
    seed=SEED,
)

fit_seconds = time.perf_counter() - t0

pred_elec = gbrt.predict(X_elec).astype(np.float64)
score_elec = np.abs(y_elec - pred_elec).astype(np.float64)

train_mse = mse(y_elec[:train_end], pred_elec[:train_end])
test_mse = mse(y_elec[train_end:], pred_elec[train_end:])

print("GBRT fit done:")
print(f"  fit seconds: {fit_seconds:.2f}")
print(f"  train MSE:   {train_mse:.6f}")
print(f"  test MSE:    {test_mse:.6f}")

In [ ]:
# =========================================================
# 3) Build data cache for evalutation of conformal methods
# =========================================================

def make_elec2_cp_data(
    X,
    y,
    pred,
    train_end,
    score=None,
    constant_eps=1e-12,
):
    """
    Build data list expected by cp_methods.

    Important:
      - is_test=True for rows after train_end.
      - cp_methods will use previous test rows as rolling calibration.
      - first R test points are naturally skipped by eval_* methods.
      - X is used only by LCP / OLCP / OLCPH for localization.
    """
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    pred = np.asarray(pred, dtype=np.float32)

    if score is None:
        score = np.abs(y - pred)
    score = np.asarray(score, dtype=np.float32)

    n = len(y)
    is_test = np.arange(n) >= int(train_end)

    y_test = y[is_test]
    const_test = (
        (np.nanmax(y_test) - np.nanmin(y_test)) < constant_eps
        if y_test.size else True
    )

    return [{
        "series_id": 0,
        "ts": np.arange(n),
        "y": y,
        "pred": pred,
        "score": score,
        "X": X,
        "is_test": is_test.astype(bool),
        "const_test": bool(const_test),
    }]


elec2_data = make_elec2_cp_data(
    X=X_elec,
    y=y_elec,
    pred=pred_elec,
    train_end=train_end,
    score=score_elec,
)

print("Built ELEC2 cp_methods data:")
print("  num series:", len(elec2_data))
print("  total rows:", len(elec2_data[0]["y"]))
print("  test rows:", int(elec2_data[0]["is_test"].sum()))
print("  X dim:", elec2_data[0]["X"].shape[1])
print("  const_test:", elec2_data[0]["const_test"])

In [ ]:
# =========================================================
# 4) Run cp_methods
# =========================================================

from cp_methods import (
    eval_cp,
    eval_lcp,
    eval_aci,
    eval_dtaci,
    eval_olcp,
    eval_olcp_adahedge,
    eval_spci,
)

METHOD_RUNNERS = {
    "cp": eval_cp,
    "lcp": eval_lcp,
    "aci": eval_aci,
    "dtaci": eval_dtaci,
    "spci": lambda data, alpha, R, exclude_const_test=False, show_pbar=True: eval_spci(
        data,
        alpha=alpha,
        R=R,
        w_lag=24,          # ELEC2 is 30-min data: 24 = 12 hours; try 48 for one day
        T_train=R,
        refit_every=24,    # refit every 12 hours; use 1 for most faithful but slower
        beta_grid_size=21,
        exclude_const_test=exclude_const_test,
        show_pbar=show_pbar,
    ),
    "olcp": eval_olcp,
    "olcph": eval_olcp_adahedge,
}

METHOD_ORDER = ["CP", "LCP", "ACI", "DTACI", "SPCI", "OLCP", "OLCPH"]

def summarize_single_series_result(df, name):
    """
    For ELEC2, there is one series_id.
    Report pooled coverage/width and finite/infinite diagnostics.
    """
    d = df.copy()
    d["width"] = d["width"].replace([np.inf, -np.inf], np.nan)

    upper = d["upper"].to_numpy(dtype=float)
    lower = d["lower"].to_numpy(dtype=float)
    isfinite_interval = np.isfinite(upper) & np.isfinite(lower)

    pct_inf = 1.0 - float(isfinite_interval.mean()) if len(d) else np.nan

    if isfinite_interval.sum() > 0:
        finite_cov = float(d.loc[isfinite_interval, "covered"].mean())
    else:
        finite_cov = np.nan

    return {
        "Method": name,
        "Coverage": float(d["covered"].mean()),
        "FiniteCoverage": finite_cov,
        "PctInf": pct_inf,
        "AvgSize": float(d["width"].mean()),
        "MedianSize": float(d["width"].median()),
        "Rows": int(len(d)),
        "FiniteSizeRows": int(np.isfinite(d["width"]).sum()),
    }


results = {}
summary_rows = []

for name, fn in METHOD_RUNNERS.items():
    df_m, sec = fn(
        elec2_data,
        alpha=ALPHA,
        R=R,
        exclude_const_test=False,
        show_pbar=True,
    )

    results[name] = df_m

    row = summarize_single_series_result(df_m, name.upper())
    row["Seconds"] = float(sec)
    row["TrainMSE"] = train_mse
    row["TestMSE"] = test_mse
    row["FitSeconds"] = fit_seconds
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

summary["Method"] = pd.Categorical(
    summary["Method"],
    categories=METHOD_ORDER,
    ordered=True,
)

summary = summary.sort_values("Method").reset_index(drop=True)

print(summary.to_string(index=False, formatters={
    "Coverage": "{:.3f}".format,
    "FiniteCoverage": "{:.3f}".format,
    "PctInf": "{:.3f}".format,
    "AvgSize": "{:.4f}".format,
    "MedianSize": "{:.4f}".format,
    "Seconds": "{:.2f}".format,
    "TrainMSE": "{:.6f}".format,
    "TestMSE": "{:.6f}".format,
    "FitSeconds": "{:.2f}".format,
}))

In [ ]:
# =========================================================
# 5) Block bootstrap SE / CI
# =========================================================

def circular_block_bootstrap_idx(n, block_len, rng):
    """
    Circular block bootstrap indices of length n.
    For ELEC2 full half-hourly data:
      block_len=48 roughly equals one day.
    """
    if n <= 0:
        return np.array([], dtype=int)

    block_len = int(min(max(1, block_len), n))
    n_blocks = int(np.ceil(n / block_len))

    starts = rng.integers(0, n, size=n_blocks)
    idx = np.concatenate([
        (s + np.arange(block_len)) % n
        for s in starts
    ])

    return idx[:n]


def _nanmean_safe(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    return float(np.mean(x))


def block_bootstrap_one_method(
    df,
    method_name,
    block_len=48,
    B=1000,
    seed=42,
    ci_alpha=0.05,
):
    """
    Bootstrap overall coverage and size for one method dataframe.
    """
    d = df.copy()
    d = d.sort_values("t") if "t" in d.columns else d

    covered = d["covered"].astype(float).to_numpy()
    width = (
        d["width"]
        .replace([np.inf, -np.inf], np.nan)
        .astype(float)
        .to_numpy()
    )

    n = len(d)
    rng = np.random.default_rng(seed)

    cov_bs = np.empty(B, dtype=float)
    size_bs = np.empty(B, dtype=float)

    for b in range(B):
        idx = circular_block_bootstrap_idx(n, block_len, rng)
        cov_bs[b] = float(np.mean(covered[idx]))
        size_bs[b] = _nanmean_safe(width[idx])

    lo_q = ci_alpha / 2
    hi_q = 1 - ci_alpha / 2

    cov_finite = cov_bs[np.isfinite(cov_bs)]
    size_finite = size_bs[np.isfinite(size_bs)]

    out = {
        "Method": method_name.upper(),

        "Coverage": float(np.mean(covered)),
        "Coverage_SE": float(np.std(cov_finite, ddof=1)) if len(cov_finite) > 1 else np.nan,

        "AvgSize": _nanmean_safe(width),
        "AvgSize_SE": float(np.std(size_finite, ddof=1)) if len(size_finite) > 1 else np.nan,

        "Rows": int(n),
        "B": int(B),
        "block_len": int(block_len),
    }

    return out, {"Coverage_bs": cov_bs, "AvgSize_bs": size_bs}


def block_bootstrap_all_methods(
    results,
    block_len=48,
    B=1000,
    seed=42,
    ci_alpha=0.05,
    show_pbar=True,
):
    rows = []
    draws = {}

    items = list(results.items())
    iterator = tqdm(items, desc="bootstrap methods") if show_pbar else items

    for i, (name, df_m) in enumerate(iterator):
        row, boot = block_bootstrap_one_method(
            df=df_m,
            method_name=name,
            block_len=block_len,
            B=B,
            seed=seed + 1000 * i,
            ci_alpha=ci_alpha,
        )
        rows.append(row)
        draws[name] = boot

    out = pd.DataFrame(rows)

    out["Method"] = pd.Categorical(
        out["Method"],
        categories=METHOD_ORDER,
        ordered=True,
    )

    out = out.sort_values("Method").reset_index(drop=True)

    return out, draws


boot_summary, boot_draws = block_bootstrap_all_methods(
    results=results,
    block_len=BOOT_BLOCK_LEN,
    B=BOOT_B,
    seed=SEED,
    ci_alpha=0.05,
    show_pbar=True,
)

print(boot_summary.to_string(index=False, formatters={
    "Coverage": "{:.3f}".format,
    "Coverage_SE": "{:.4f}".format,
    "AvgSize": "{:.4f}".format,
    "AvgSize_SE": "{:.4f}".format,
}))